# Raw Data Exploration — SolarCycle Hackathon

Visualises the five raw public datasets in `data-real/`:

| # | File | Key patterns |
|---|------|--------------|
| 1 | `cer_sgu_solar_postcode.csv` | Victorian install boom, end-of-life cohort, postcode concentration |
| 2 | `vic_waste_infrastructure.csv` | Facility type mix, geographic spread |
| 3 | `solar_vic_lga.xlsx` | LGA rankings, residential vs business vs rental |
| 4 | `solar_vic_pv_modules.xlsx` | Top panel brands, capacity tiers, quarterly trend |
| 5 | `solar_vic_inverters.xlsx` | Top inverter brands, rated power distribution, quarterly trend |

Run all cells top-to-bottom. Requires: `pandas`, `matplotlib`, `seaborn`, `openpyxl`.

In [ ]:
import re
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 110

DATA = '../'  # paths relative to this notebook

---
## 1 · CER Postcode Solar Installs (`cer_sgu_solar_postcode.csv`)

Each row is a postcode. Columns are monthly install counts from Jan 2011, plus a pre-2011 historic total.
We filter to **Victorian postcodes (3000–3999)** throughout.

In [ ]:
cer_raw = pd.read_csv(DATA + 'cer_sgu_solar_postcode.csv', dtype={'Small Unit Installation Postcode': str})
cer_raw.columns = cer_raw.columns.str.strip()

# filter Victorian postcodes
vic = cer_raw[cer_raw['Small Unit Installation Postcode'].str.match(r'^3\d{3}$', na=False)].copy()
vic = vic.rename(columns={'Small Unit Installation Postcode': 'postcode',
                           'Historic Total Installation Quantity (2001 - 2010)': 'pre2011',
                           'Total Installation Quantity': 'total'})

# identify the monthly columns
month_cols = [c for c in vic.columns if re.match(r'^[A-Z][a-z]{2} \d{4} - Installation Quantity$', c)]
print(f'Victorian postcodes: {len(vic):,}   Monthly columns: {len(month_cols)}')
vic[['postcode', 'pre2011', 'total']].head(3)

In [ ]:
# --- Yearly installs across all Victorian postcodes ---
def col_year(col):
    m = re.search(r'(\d{4})', col)
    return int(m.group(1)) if m else None

by_year = (
    vic[month_cols]
    .sum()               # total per month across all VIC postcodes
    .rename(lambda c: col_year(c))
    .groupby(level=0)
    .sum()
    .reset_index()
)
by_year.columns = ['year', 'installs']

# add the pre-2011 aggregate as a single 2001-2010 bar
pre2011_total = int(vic['pre2011'].sum())

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(by_year['year'], by_year['installs'], color=sns.color_palette()[0])
ax.axhline(pre2011_total / 10, color='salmon', linestyle='--', linewidth=1.2,
           label=f'Pre-2011 avg/yr (~{pre2011_total//10:,})')
ax.set_title('Victorian Solar Installs per Year  (CER, 2011–2026)', fontsize=13)
ax.set_xlabel('Year')
ax.set_ylabel('Number of installations')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()
print(f'Peak year: {by_year.loc[by_year.installs.idxmax(), "year"]}  '
      f'({by_year.installs.max():,.0f} installs)')

In [ ]:
# --- Top 25 Victorian postcodes by all-time total ---
top25 = vic.nlargest(25, 'total')[['postcode', 'total', 'pre2011']].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=top25, x='postcode', y='total', ax=ax, color=sns.color_palette()[1])
ax.set_title('Top 25 Victorian Postcodes by Total Solar Installs', fontsize=13)
ax.set_xlabel('Postcode')
ax.set_ylabel('Total installations')
ax.tick_params(axis='x', rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# --- End-of-life cohort: pre-2011 installations by postcode (top 30) ---
# Systems installed before 2011 are 15+ years old — high end-of-life risk.
eol = vic[vic['pre2011'] > 0].nlargest(30, 'pre2011')[['postcode', 'pre2011']]

fig, ax = plt.subplots(figsize=(11, 4))
sns.barplot(data=eol, x='postcode', y='pre2011', ax=ax, color='coral')
ax.set_title('Top 30 Postcodes: Pre-2011 Installs (End-of-Life Cohort)', fontsize=13)
ax.set_xlabel('Postcode')
ax.set_ylabel('Pre-2011 installations')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()
print(f'Total pre-2011 VIC installs: {pre2011_total:,}  '
      f'(these are 15+ years old — prime end-of-life collection targets)')

In [ ]:
# --- Monthly seasonality: average installs per calendar month (all VIC postcodes) ---
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
month_sums = vic[month_cols].sum()  # total across all VIC postcodes

seasonal = (
    month_sums
    .rename(lambda c: re.match(r'^([A-Z][a-z]{2})', c).group(1))
    .groupby(level=0)
    .mean()              # average monthly across all years
    .reindex(month_order)
    .reset_index()
)
seasonal.columns = ['month', 'avg_installs']

fig, ax = plt.subplots(figsize=(9, 3.5))
sns.barplot(data=seasonal, x='month', y='avg_installs', ax=ax, color=sns.color_palette()[2])
ax.set_title('Average Monthly Solar Installs by Calendar Month (VIC)', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Average monthly installs')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

---
## 2 · Victoria Waste Infrastructure (`vic_waste_infrastructure.csv`)

Each row is one real facility in Victoria. Columns: Facility Name, Owner, Facility Type, Infrastructure Type, Address, Suburb, LGA, Latitude, Longitude.

In [ ]:
waste = pd.read_csv(DATA + 'vic_waste_infrastructure.csv')
waste.columns = waste.columns.str.strip()
print(f'Facilities: {len(waste):,}   Columns: {list(waste.columns)}')
waste[['Facility Name','Facility Type','Infrastructure Type','Suburb','LGA']].head(4)

In [ ]:
# --- Facility Type breakdown ---
ftype = waste['Facility Type'].value_counts().reset_index()
ftype.columns = ['facility_type', 'count']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# bar: facility type
sns.barplot(data=ftype, y='facility_type', x='count', ax=axes[0], palette='tab10')
axes[0].set_title('Count by Facility Type', fontsize=12)
axes[0].set_xlabel('Number of facilities')
axes[0].set_ylabel('')

# bar: infrastructure type (top 15)
infra = waste['Infrastructure Type'].value_counts().head(15).reset_index()
infra.columns = ['infrastructure_type', 'count']
sns.barplot(data=infra, y='infrastructure_type', x='count', ax=axes[1],
            color=sns.color_palette()[3])
axes[1].set_title('Top 15 Infrastructure Types', fontsize=12)
axes[1].set_xlabel('Number of facilities')
axes[1].set_ylabel('')

plt.suptitle('Victoria Waste & Resource Recovery Facilities', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- Geographic scatter of facilities coloured by Facility Type ---
geo = waste.dropna(subset=['Latitude', 'Longitude']).copy()
geo['Latitude'] = pd.to_numeric(geo['Latitude'], errors='coerce')
geo['Longitude'] = pd.to_numeric(geo['Longitude'], errors='coerce')
geo = geo.dropna(subset=['Latitude', 'Longitude'])

types = geo['Facility Type'].unique()
palette = dict(zip(types, sns.color_palette('tab10', len(types))))

fig, ax = plt.subplots(figsize=(9, 7))
for ftype_name, group in geo.groupby('Facility Type'):
    ax.scatter(group['Longitude'], group['Latitude'],
               label=ftype_name, s=18, alpha=0.7, color=palette[ftype_name])
ax.set_title('Waste & Recovery Facilities — Victoria (coloured by type)', fontsize=13)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()
print(f'Facilities plotted: {len(geo):,} / {len(waste):,}')

---
## 3 · Solar Victoria LGA Installs (`solar_vic_lga.xlsx`)

Rows are Victorian LGAs. Columns count Solar Victoria program installs by type (Business, Owner-Occupier, Rental, Battery, Hot Water, Heating).

In [ ]:
lga_raw = pd.read_excel(DATA + 'solar_vic_lga.xlsx')
lga_raw.columns = lga_raw.columns.str.strip()
print(f'LGA rows: {len(lga_raw):,}   Columns: {list(lga_raw.columns)}')
lga_raw.head(3)

In [ ]:
# Identify PV columns (Business, Owner Occupier, Rental) — handle name variants
pv_cols = [c for c in lga_raw.columns if 'Solar PV' in c or 'solar pv' in c.lower()]
print('PV columns detected:', pv_cols)

lga = lga_raw.copy()
lga_name_col = lga.columns[0]  # first column is the LGA name

if pv_cols:
    lga['total_solar_pv'] = lga[pv_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1)
else:
    # fallback: sum all numeric columns
    numeric_cols = lga.select_dtypes(include='number').columns.tolist()
    lga['total_solar_pv'] = lga[numeric_cols].sum(axis=1)

top_lga = lga.nlargest(20, 'total_solar_pv')[[lga_name_col, 'total_solar_pv'] + pv_cols].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=top_lga, x=lga_name_col, y='total_solar_pv', ax=ax, color=sns.color_palette()[4])
ax.set_title('Top 20 LGAs by Total Solar PV Installs (Solar Victoria program)', fontsize=13)
ax.set_xlabel('LGA')
ax.set_ylabel('Total Solar PV installations')
ax.tick_params(axis='x', rotation=60)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# --- Residential vs Business vs Rental breakdown for top 15 LGAs ---
if len(pv_cols) >= 3:
    seg_cols = pv_cols[:3]  # Business, Owner Occupier, Rental
    seg = top_lga.head(15).set_index(lga_name_col)[seg_cols]
    seg.columns = [c.replace('Solar PV ', '').replace('(', '').replace(')', '') for c in seg.columns]

    seg.plot(kind='bar', stacked=True, figsize=(12, 5),
             color=sns.color_palette('Set2', len(seg.columns)))
    plt.title('Solar PV Installs by Segment — Top 15 LGAs (Solar Victoria)', fontsize=13)
    plt.xlabel('LGA')
    plt.ylabel('Installations')
    plt.xticks(rotation=50, ha='right')
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.show()
else:
    print('Fewer than 3 PV columns found; skipping segment breakdown.')

---
## 4 · Solar Victoria PV Module Models (`solar_vic_pv_modules.xlsx`)

Each row is one panel model. Quarter columns count modules installed through Solar Victoria programs.

In [ ]:
pv_raw = pd.read_excel(DATA + 'solar_vic_pv_modules.xlsx')
pv_raw.columns = pv_raw.columns.str.strip()
print(f'PV module rows: {len(pv_raw):,}   Columns: {list(pv_raw.columns[:8])} ...')

# Identify quarter columns (pattern: YYYY Q[1-4])
q_cols_pv = [c for c in pv_raw.columns if re.match(r'^\d{4} Q[1-4]$', str(c))]
print(f'Quarter columns: {len(q_cols_pv)}  first={q_cols_pv[0] if q_cols_pv else None}  last={q_cols_pv[-1] if q_cols_pv else None}')

pv_raw[q_cols_pv] = pv_raw[q_cols_pv].apply(pd.to_numeric, errors='coerce').fillna(0)
pv_raw['total'] = pv_raw[q_cols_pv].sum(axis=1)
pv_raw.head(3)

In [ ]:
# --- Top 15 PV manufacturers by total modules ---
brand_col = [c for c in pv_raw.columns if 'brand' in c.lower()]
brand_col = brand_col[0] if brand_col else pv_raw.columns[0]

top_brands = (
    pv_raw.groupby(brand_col)['total']
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
)
top_brands.columns = ['brand', 'total_modules']

fig, ax = plt.subplots(figsize=(11, 4))
sns.barplot(data=top_brands, x='brand', y='total_modules', ax=ax, palette='Blues_r')
ax.set_title('Top 15 PV Module Brands — Solar Victoria Program', fontsize=13)
ax.set_xlabel('Brand')
ax.set_ylabel('Total modules installed')
ax.tick_params(axis='x', rotation=60)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# --- Quarterly install trend for PV modules ---
quarterly_pv = pv_raw[q_cols_pv].sum().reset_index()
quarterly_pv.columns = ['quarter', 'modules']

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(range(len(quarterly_pv)), quarterly_pv['modules'], marker='o', markersize=4,
        color=sns.color_palette()[5])
ax.fill_between(range(len(quarterly_pv)), quarterly_pv['modules'], alpha=0.15,
                color=sns.color_palette()[5])
step = max(1, len(quarterly_pv) // 12)
ax.set_xticks(range(0, len(quarterly_pv), step))
ax.set_xticklabels(quarterly_pv['quarter'].iloc[::step], rotation=45, ha='right', fontsize=8)
ax.set_title('PV Modules Installed per Quarter — Solar Victoria Program', fontsize=13)
ax.set_xlabel('Quarter')
ax.set_ylabel('Modules installed')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# --- Panel capacity distribution (weighted by install count) ---
cap_col = [c for c in pv_raw.columns if 'capacity' in c.lower() or '(w)' in c.lower() or 'watt' in c.lower()]
cap_col = cap_col[0] if cap_col else None

if cap_col:
    pv_raw[cap_col] = pd.to_numeric(pv_raw[cap_col], errors='coerce')
    # weight each model by its total install count
    cap_weighted = pv_raw.dropna(subset=[cap_col])[[ cap_col, 'total']]
    cap_weighted = cap_weighted[cap_weighted['total'] > 0]

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(cap_weighted[cap_col], weights=cap_weighted['total'],
            bins=30, color=sns.color_palette()[6], edgecolor='white')
    ax.set_title(f'PV Module Capacity Distribution (weighted by installs)\nColumn: {cap_col}', fontsize=12)
    ax.set_xlabel('Capacity (W)')
    ax.set_ylabel('Total modules installed')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    plt.tight_layout()
    plt.show()
    med = (cap_weighted[cap_col] * cap_weighted['total']).sum() / cap_weighted['total'].sum()
    print(f'Weighted median-ish capacity: ~{med:.0f} W')
else:
    print('No capacity column found in PV module file.')

---
## 5 · Solar Victoria Inverter Models (`solar_vic_inverters.xlsx`)

Each row is one inverter model. Quarter columns count inverters installed through Solar Victoria programs.

In [ ]:
inv_raw = pd.read_excel(DATA + 'solar_vic_inverters.xlsx')
inv_raw.columns = inv_raw.columns.str.strip()
print(f'Inverter rows: {len(inv_raw):,}   Columns: {list(inv_raw.columns[:8])} ...')

q_cols_inv = [c for c in inv_raw.columns if re.match(r'^\d{4} Q[1-4]$', str(c))]
inv_raw[q_cols_inv] = inv_raw[q_cols_inv].apply(pd.to_numeric, errors='coerce').fillna(0)
inv_raw['total'] = inv_raw[q_cols_inv].sum(axis=1)
inv_raw.head(3)

In [ ]:
# --- Top 15 inverter brands ---
inv_brand_col = [c for c in inv_raw.columns if 'brand' in c.lower()]
inv_brand_col = inv_brand_col[0] if inv_brand_col else inv_raw.columns[0]

top_inv = (
    inv_raw.groupby(inv_brand_col)['total']
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
)
top_inv.columns = ['brand', 'total_inverters']

fig, ax = plt.subplots(figsize=(11, 4))
sns.barplot(data=top_inv, x='brand', y='total_inverters', ax=ax, palette='Oranges_r')
ax.set_title('Top 15 Inverter Brands — Solar Victoria Program', fontsize=13)
ax.set_xlabel('Brand')
ax.set_ylabel('Total inverters installed')
ax.tick_params(axis='x', rotation=60)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# --- Quarterly trend and rated power distribution side by side ---
quarterly_inv = inv_raw[q_cols_inv].sum().reset_index()
quarterly_inv.columns = ['quarter', 'inverters']

power_col = [c for c in inv_raw.columns if 'power' in c.lower() or '(va' in c.lower() or 'rated' in c.lower()]
power_col = power_col[0] if power_col else None

has_power = power_col is not None
ncols = 2 if has_power else 1
fig, axes = plt.subplots(1, ncols, figsize=(14 if has_power else 9, 4))
if ncols == 1:
    axes = [axes]

# quarterly trend
ax = axes[0]
ax.plot(range(len(quarterly_inv)), quarterly_inv['inverters'], marker='o', markersize=4,
        color='darkorange')
ax.fill_between(range(len(quarterly_inv)), quarterly_inv['inverters'], alpha=0.15, color='darkorange')
step = max(1, len(quarterly_inv) // 12)
ax.set_xticks(range(0, len(quarterly_inv), step))
ax.set_xticklabels(quarterly_inv['quarter'].iloc[::step], rotation=45, ha='right', fontsize=8)
ax.set_title('Inverters Installed per Quarter — Solar Victoria', fontsize=12)
ax.set_ylabel('Inverters installed')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# rated power distribution
if has_power:
    ax2 = axes[1]
    inv_raw[power_col] = pd.to_numeric(inv_raw[power_col], errors='coerce')
    pw = inv_raw.dropna(subset=[power_col])
    pw = pw[pw['total'] > 0]
    ax2.hist(pw[power_col] / 1000, weights=pw['total'],  # convert VA to kVA
             bins=30, color='sienna', edgecolor='white')
    ax2.set_title(f'Inverter Rated Power Distribution (weighted)\nColumn: {power_col}', fontsize=12)
    ax2.set_xlabel('Rated AC power (kVA)')
    ax2.set_ylabel('Total inverters installed')
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.show()

---
## Summary

| Dataset | Key insight |
|---------|-------------|
| CER postcode | Install boom peaked ~2012 and again 2018-2022; pre-2011 cohort now 15+ years old |
| Waste infra | Transfer stations dominate; reprocessors are sparse — solar-specific capacity is missing |
| Solar Vic LGA | Outer-suburban growth corridors dominate (Wyndham, Casey, Melton) |
| PV modules | Market concentrated in ~5 brands; capacity trending toward 400-530 W panels |
| Inverters | Concentrated in ~3-4 brands; most residential systems are 5-10 kVA range |